# 05. Simulated patient profiles (Figure 3, Supplementary Text 1)

Predicted survival curves for five simulated (non-real) patient profiles.

* **A** 66-year-old woman, BMI 26, asymptomatic, diabetes, never-smoker; FVC 4.0 L, FEV1 3.15 L, DLCO 20.0;
  R0 wedge resection, no neoadjuvant/adjuvant therapy; papillary adenocarcinoma, **T1b** N0 M0;
  VEGF 4+, EGFR 3+, Ki-67 1+, p53 1+.
* **B** 65-year-old man, BMI 23.5, exertional dyspnea, hypertension and chronic liver disease; ex-smoker
  (0.5 pack/day for 40 years, quit 2 years earlier); FVC 3.7 L, FEV1 2.4 L, DLCO 15.0; R0 simple lobectomy,
  adjuvant chemotherapy; acinar adenocarcinoma, T2a N1 M0; VEGF 3+, EGFR 1+, Ki-67 0, p53 0.
* **C** 70-year-old man, BMI 28, cough and hemoptysis, coronary artery disease; ex-smoker (1 pack/day for
  40 years, quit 10 years earlier); FVC 3.5 L, FEV1 2.1 L, DLCO 15.0; R0 simple lobectomy, adjuvant
  chemoradiotherapy; solid adenocarcinoma, T2a N2 M0; EGFR 2+, other markers negative.
* **D** identical to A except non-R0 resection.
* **E** identical to B except no adjuvant therapy.

Coding of the input variables is described in the manuscript (Supplementary Table 1).

The differences between Patients A and D and between Patients B and E are differences in model predictions and should not be interpreted as treatment effects.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore", category=FutureWarning)

OUTPUT_DIR   = Path("../outputs")
PREPROC_PATH = OUTPUT_DIR / "preprocessor.joblib"  # written by 01_data_preprocessor
MODEL_PATH   = OUTPUT_DIR / "RSF_model.joblib"     # written by 02_RSF_train

In [ ]:
preprocessor = joblib.load(PREPROC_PATH)
model        = joblib.load(MODEL_PATH)

In [ ]:
BASE_A = {
    "Age": 66, "Gender": 2, "BMI": 26,
    **{f"symptom_{i}": 0 for i in range(1, 13)},
    "DM": 1, "HTN": 0, "CAD": 0, "Hepatic Disease": 0, "Renal Disease": 0,
    "Respiratory Disease": 0, "Malignancy": 0,
    "Smoke Hx": 0, "Smoke Quit_days": -1, "Smoke Amount": -1, "Smoke Duration": -1,
    "Spirometry FVC": 4.0, "Spirometry FEV1": 3.15, "Diffusion DLCO": 20.0,
    "Neoadjuvant therapy": 0, "Opextent": 1, "resection type": 1, "Postop Adjuvant Therapy": 0,
    "Adeno type": 2, "T_stage": 5, "N_stage": 1, "M_stage": 1,
    "VEGF": 4, "EGFR": 3, "Ki-67": 1, "p53": 1,
}
BASE_B = {
    "Age": 65, "Gender": 1, "BMI": 23.5,
    **{f"symptom_{i}": 0 for i in range(1, 13)}, "symptom_4": 1,
    "DM": 0, "HTN": 1, "CAD": 0, "Hepatic Disease": 1, "Renal Disease": 0,
    "Respiratory Disease": 0, "Malignancy": 0,
    "Smoke Hx": 1, "Smoke Quit_days": 730, "Smoke Amount": 0.5, "Smoke Duration": 40,
    "Spirometry FVC": 3.7, "Spirometry FEV1": 2.4, "Diffusion DLCO": 15,
    "Neoadjuvant therapy": 0, "Opextent": 3, "resection type": 1, "Postop Adjuvant Therapy": 1,
    "Adeno type": 1, "T_stage": 7, "N_stage": 2, "M_stage": 1,
    "VEGF": 3, "EGFR": 1, "Ki-67": 0, "p53": 0,
}
BASE_C = {
    "Age": 70, "Gender": 1, "BMI": 28,
    **{f"symptom_{i}": 0 for i in range(1, 13)}, "symptom_1": 1, "symptom_2": 1,
    "DM": 0, "HTN": 0, "CAD": 1, "Hepatic Disease": 0, "Renal Disease": 0,
    "Respiratory Disease": 0, "Malignancy": 0,
    "Smoke Hx": 1, "Smoke Quit_days": 3650, "Smoke Amount": 1, "Smoke Duration": 40,
    "Spirometry FVC": 3.5, "Spirometry FEV1": 2.1, "Diffusion DLCO": 15,
    "Neoadjuvant therapy": 0, "Opextent": 3, "resection type": 1, "Postop Adjuvant Therapy": 3,
    "Adeno type": 4, "T_stage": 7, "N_stage": 3, "M_stage": 1,
    "VEGF": 0, "EGFR": 2, "Ki-67": 0, "p53": 0,
}
PATIENTS = {
    "Patient A": (BASE_A, "tab:blue"),
    "Patient B": (BASE_B, "tab:green"),
    "Patient C": (BASE_C, "tab:purple"),
    "Patient D": ({**BASE_A, "resection type": 2}, "tab:orange"),          # A with non-R0 resection
    "Patient E": ({**BASE_B, "Postop Adjuvant Therapy": 0}, "tab:pink"),  # B without adjuvant therapy
}

In [ ]:
YEARS = (2, 5, 7, 10)
fig, ax = plt.subplots(figsize=(7, 5))
handles, labels, table = [], [], {}
for name, (inputs, color) in PATIENTS.items():
    sf = model.predict_survival_function(preprocessor.transform(pd.DataFrame([inputs])),
                                         return_array=False)[0]
    (h,) = ax.step(sf.x / 365.25, sf.y, where="post", color=color)
    surv = {}
    for yr in YEARS:
        i = np.searchsorted(sf.x, yr * 365, side="right") - 1
        surv[yr] = sf.y[i]
        ax.axvline(yr, color="gray", linestyle=":", linewidth=0.8, zorder=1)
        ax.scatter(yr, sf.y[i], color="crimson", s=25, zorder=3)
    table[name] = {f"{yr}-year": round(100 * p, 1) for yr, p in surv.items()}
    handles.append(h)
    labels.append(f"{name}: " + " -> ".join(f"{100 * p:.1f}%" for p in surv.values()))

ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("Years after surgery (0-10)"); ax.set_ylabel("Survival probability (%)")
ax.set_xticks(range(0, 11)); ax.set_xlim(0, 10.5); ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(handles, labels, loc="lower left", fontsize=9, framealpha=0.8)
ax.set_title("Predicted survival curves")
plt.tight_layout()
# plt.savefig("Figure3_simulated_patients.jpg", dpi=300, bbox_inches="tight")
plt.show()
print(pd.DataFrame(table).T.to_string())